# v2 — 01: Data Gathering
**Iteration 2** extends v1 data with:
- NFL draft pick data (draft capital, age at draft)
- NFL combine measurements (athleticism profile)
- Weekly injury reports (aggregated to games missed per season)
- College season stats via College Football Data API (final year before draft)

**Prerequisite:** Run `v1_01_data_gathering.ipynb` first for base NFL/Sleeper data.

All fetchers use per-season checkpointing — safe to interrupt and resume.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.fetchers.nfl_fetcher import fetch_draft_picks, fetch_combine_data, fetch_injuries
from src.fetchers.college_fetcher import fetch_college_stats
from src.pipeline.cleaner import load_to_db

import pandas as pd
pd.set_option('display.max_columns', 40)

## 1. Draft Picks
One row per drafted player per season. Key fields: `player_id` (gsis_id), `pick`, `round`, `age`, `cfb_player_id`.

In [ ]:
draft_picks = fetch_draft_picks(seasons=list(range(2016, 2025)))
print(f'Shape: {draft_picks.shape}')
print('Columns:', draft_picks.columns.tolist())
draft_picks.head(3)

## 2. Combine Data
Athletic measurements per prospect. Joined to player data via `pfr_id`.

In [ ]:
combine_data = fetch_combine_data(seasons=list(range(2016, 2025)))
print(f'Shape: {combine_data.shape}')
print('Columns:', combine_data.columns.tolist())
combine_data.head(3)

In [ ]:
# Combine attendance rate — many players skip the combine
for col in ['forty', 'vertical', 'bench']:
    if col in combine_data.columns:
        pct = combine_data[col].notna().mean()
        print(f'  {col}: {pct:.1%} have measurement')

## 3. Injury Reports
Weekly injury designations aggregated to season-level: `games_missed` and `ir_flag` per player per season.

In [ ]:
injuries = fetch_injuries(seasons=list(range(2016, 2025)))
print(f'Shape: {injuries.shape}')
injuries.head(3)

In [ ]:
# Injury summary
print('Avg games missed per player-season:', injuries['games_missed'].mean().round(2))
print('Players with IR flag any season:', injuries['ir_flag'].sum())

## 4. College Stats
Final college season production for each draft class (college year = draft year - 1).
Uses College Football Data API — may take a few minutes due to rate limiting.

In [ ]:
college_stats = fetch_college_stats(draft_seasons=list(range(2016, 2025)))
print(f'Shape: {college_stats.shape}')
print('Columns:', college_stats.columns.tolist())
college_stats.head(3)

In [ ]:
# Top college receivers by draft year
if not college_stats.empty and 'college_rec_yards' in college_stats.columns:
    top_receivers = (
        college_stats.sort_values('college_rec_yards', ascending=False)
        [['player_name', 'team', 'draft_season', 'college_rec_yards',
          'college_rec_tds', 'college_dominator_rate']]
        .head(15)
    )
    print('Top college receivers (2015-2024):')
    print(top_receivers.to_string(index=False))

## 5. Load to SQLite

In [ ]:
load_to_db(draft_picks, 'draft_picks')
load_to_db(combine_data, 'combine_data')
load_to_db(injuries, 'injuries')
if not college_stats.empty:
    load_to_db(college_stats, 'college_stats')
print('All v2 tables loaded to fantasy.db')

## 6. Sanity Checks

In [ ]:
print('Draft picks seasons:', sorted(draft_picks['season'].unique()) if 'season' in draft_picks.columns else 'N/A')
print('Combine seasons:', sorted(combine_data['season'].unique()) if 'season' in combine_data.columns else 'N/A')
print('Injury seasons:', sorted(injuries['season'].unique()) if 'season' in injuries.columns else 'N/A')

# Check cfb_player_id join key exists
if 'cfb_player_id' in draft_picks.columns:
    pct = draft_picks['cfb_player_id'].notna().mean()
    print(f'Draft picks with cfb_player_id: {pct:.1%} (join key for college stats)')